# nbplay: from one widget to a session

Click any widget once before playing: browsers keep audio off until a user gesture.

## 1. A synth on its own

One Python object, one element in the browser. Its `config` is a pydantic model that Python owns; its `status` is owned by the browser.

In [ ]:
from nbplay import SynthWidget

synth = SynthWidget.from_preset("Acid bass")
synth

**Python → browser:** assign a field and the knob moves.

In [ ]:
synth.config.oscillator_type = "square"
synth.config.cutoff = 5000

Every write is validated, whether it comes from Python or from the browser.

In [ ]:
try:
    synth.config.cutoff = 100_000
except ValueError as error:
    print(error.errors()[0]["msg"])

**Browser → Python:** turn a knob or press play in the widget above, and the listener prints the change.

In [ ]:
stop_listening = synth.on_change(lambda widget, name, old, new: print(f"{name}: {old!r} -> {new!r}"),
                                 names=("cutoff", "resonance", "oscillator_type", "is_playing"))

`is_playing` belongs to the browser, so Python reads it but can't assign it. `play()` sends a command; the browser starts the sound and reports the new status.

In [ ]:
synth.play();

In [ ]:
synth.status.is_playing  # True once the browser has started it

In [ ]:
synth.stop()
stop_listening()

## 2. A sampler on its own

A plucked string, generated in Python and loaded as the sample. Click the pads to play it.

In [ ]:
import numpy as np

def pluck(seconds=1.2, rate=44100, hz=220.0):
    """A Karplus-Strong plucked string."""
    period = round(rate / hz)
    ring = np.random.uniform(-1, 1, period)
    out = np.empty(int(seconds * rate))
    for i in range(len(out)):
        j = i % period
        out[i] = ring[j]
        ring[j] = 0.996 * 0.5 * (ring[j] + ring[(j + 1) % period])
    return out

from nbplay import SamplerWidget

sampler = SamplerWidget().load_sample(pluck(), sample_rate=44100, root_note=57, name="pluck")
sampler.configure_pads(pad_notes=[45, 48, 52, 55, 57, 60, 64, 67])
sampler

## 3. They don't know about each other

Nothing in Python connects these two objects, and in the browser each element runs on its own private clock because neither has a `session_id`.

In [ ]:
synth.config.session_id, sampler.config.session_id

## 4. Put them in a session

A `Session` is an ordinary Python object. It creates a transport, mixer, timeline, launcher and MIDI router, and gives them all one `session_id`.

In [ ]:
from nbplay import Session, SequencerWidget, DrumMachineWidget

session = Session(bpm=112)
session.transport

`add_track` adds a mixer channel and writes `session_id` and `channel_index` into the widgets we pass, including the synth and sampler from above.

In [ ]:
bass = session.add_track("Bass", SequencerWidget(), synth)
pluck_track = session.add_track("Pluck", SequencerWidget(view="rows", num_voices=3), sampler)
beat = session.add_track("Beat", None, DrumMachineWidget())

for w in session.widgets:
    channel = w.config.channel_index if "channel_index" in w.CONFIG else ""
    print(f"{type(w).__name__:18} {w.config.session_id}  {channel}")

**That is the whole link.**

- In Python, `session` holds references to the widgets and registers `on_change` listeners on them.
- In the browser, each element reads `session_id` from its own config and joins `globalThis.__nbplay[session_id]`: one `AudioContext`, one clock, one mixer graph, and an instrument per track.

The synth and sampler displayed above just joined the session too: the patch changed their `sessionId`, and their elements moved onto the shared bus.

In [ ]:
beat.sound_source.set_row(0, [0, 4, 8, 12])         # kick
beat.sound_source.set_row(1, [4, 12])               # snare
beat.sound_source.set_row(2, range(0, 16, 2), 70)   # closed hat

bass.sequencer.set_row_note(0, 36)
bass.sequencer.set_row(0, [0, 3, 6, 10, 12])

for row, note in enumerate([57, 60, 64]):
    pluck_track.sequencer.set_row_note(row, note)
pluck_track.sequencer.set_row(0, [0, 6])
pluck_track.sequencer.set_row(1, [2, 8, 12])
pluck_track.sequencer.set_row(2, [4, 10, 14])

In [ ]:
session.mixer

In [ ]:
beat.sound_source

In [ ]:
bass.sequencer

In [ ]:
pluck_track.sequencer

## 5. One clock

`play()` is one command to the transport. The transport starts the shared clock in the browser, and every sequencer and the drum machine starts on the same beat.

In [ ]:
session.play();

Setting the tempo in Python loops over `session.widgets` and writes each config. Each element then pushes the tempo to the shared clock (only the first push changes it).

In [ ]:
def tempos():
    for w in session.widgets:
        if "bpm" in w.CONFIG:
            print(f"{type(w).__name__:18} {w.config.bpm}")

session.bpm = 128
tempos()

Now drag the tempo on the transport above and run the next cell. The browser changes the clock immediately; the transport's new `bpm` comes back to Python, and the session's listener copies it to every widget.

In [ ]:
tempos()

In [ ]:
session.stop();

## 6. The same session as an app

The served app hosts the same kind of session, with one WebSocket per widget. Run this in a terminal for the demo session:

```bash
python -m nbplay.app
```